# Week 12: MCP & Observability

**Lab companion to [week_12.md](../agenda/week_12.md)**

In this lab, you will:
1. Understand Model Context Protocol (MCP)
2. Implement logging and tracing
3. Monitor LLM performance
4. Debug AI applications

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import time
from datetime import datetime

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Basic Logging

In [ ]:
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("ai_app")

def logged_completion(messages: list, **kwargs) -> dict:
    """LLM call with logging."""

    # Log request
    logger.info(f"Request: {len(messages)} messages")

    start = time.time()

    try:
        response = client.chat.completions.create(
            model=kwargs.get("model", "gpt-4o-mini"),
            messages=messages,
            **{k: v for k, v in kwargs.items() if k != "model"}
        )

        duration = time.time() - start

        # Log response
        logger.info(f"Response: {response.usage.total_tokens} tokens in {duration:.2f}s")

        return {
            "content": response.choices[0].message.content,
            "tokens": response.usage.total_tokens,
            "duration": duration
        }

    except Exception as e:
        logger.error(f"Error: {e}")
        raise

# Test
result = logged_completion([{"role": "user", "content": "Hi!"}])
print(f"\nResponse: {result['content']}")

## 2. Structured Tracing

In [ ]:
import uuid
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class Trace:
    trace_id: str
    span_id: str
    parent_span_id: Optional[str]
    name: str
    start_time: float
    end_time: Optional[float] = None
    attributes: dict = None
    status: str = "running"

    def finish(self, status: str = "success"):
        self.end_time = time.time()
        self.status = status

    @property
    def duration(self) -> float:
        if self.end_time:
            return self.end_time - self.start_time
        return 0.0

class Tracer:
    def __init__(self):
        self.traces = []
        self.current_trace_id = None
        self.current_span_id = None

    def start_trace(self, name: str, attributes: dict = None) -> Trace:
        trace_id = str(uuid.uuid4())
        span_id = str(uuid.uuid4())

        trace = Trace(
            trace_id=trace_id,
            span_id=span_id,
            parent_span_id=self.current_span_id,
            name=name,
            start_time=time.time(),
            attributes=attributes or {}
        )

        self.traces.append(trace)
        self.current_trace_id = trace_id
        self.current_span_id = span_id

        return trace

    def start_span(self, name: str, attributes: dict = None) -> Trace:
        span_id = str(uuid.uuid4())

        trace = Trace(
            trace_id=self.current_trace_id,
            span_id=span_id,
            parent_span_id=self.current_span_id,
            name=name,
            start_time=time.time(),
            attributes=attributes or {}
        )

        self.traces.append(trace)
        self.current_span_id = span_id

        return trace

    def summary(self) -> list:
        return [
            {
                "name": t.name,
                "duration": f"{t.duration:.3f}s",
                "status": t.status,
                **t.attributes
            }
            for t in self.traces
        ]

tracer = Tracer()
print("Tracer ready!")

In [ ]:
# Example traced RAG pipeline
def traced_rag(question: str) -> str:
    # Start main trace
    main_trace = tracer.start_trace("rag_query", {"question": question})

    # Embedding span
    embed_span = tracer.start_span("embed_query")
    # Simulate embedding
    time.sleep(0.1)
    embed_span.finish()

    # Retrieval span
    retrieve_span = tracer.start_span("retrieve_docs", {"top_k": 3})
    # Simulate retrieval
    time.sleep(0.1)
    docs = ["Doc 1", "Doc 2", "Doc 3"]
    retrieve_span.attributes["docs_found"] = len(docs)
    retrieve_span.finish()

    # Generation span
    gen_span = tracer.start_span("generate_answer")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        max_tokens=50
    )
    gen_span.attributes["tokens"] = response.usage.total_tokens
    gen_span.finish()

    main_trace.finish()

    return response.choices[0].message.content

# Run
answer = traced_rag("What is Python?")
print(f"Answer: {answer}\n")

# Show traces
print("Traces:")
for trace in tracer.summary():
    print(f"  {trace}")

## 3. Metrics Collection

In [ ]:
from collections import defaultdict
import statistics

class MetricsCollector:
    """Collect and aggregate metrics."""

    def __init__(self):
        self.metrics = defaultdict(list)

    def record(self, name: str, value: float, tags: dict = None):
        """Record a metric value."""
        self.metrics[name].append({
            "value": value,
            "timestamp": time.time(),
            "tags": tags or {}
        })

    def summary(self, name: str) -> dict:
        """Get summary statistics for a metric."""
        values = [m["value"] for m in self.metrics[name]]

        if not values:
            return {}

        return {
            "count": len(values),
            "mean": statistics.mean(values),
            "min": min(values),
            "max": max(values),
            "p50": statistics.median(values),
            "p95": statistics.quantiles(values, n=20)[18] if len(values) >= 20 else max(values)
        }

    def report(self) -> dict:
        """Get all metrics summaries."""
        return {
            name: self.summary(name)
            for name in self.metrics
        }

metrics = MetricsCollector()

In [ ]:
# Instrumented LLM call
def instrumented_call(messages: list, model: str = "gpt-4o-mini") -> str:
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    duration = time.time() - start

    # Record metrics
    metrics.record("llm.latency", duration, {"model": model})
    metrics.record("llm.tokens", response.usage.total_tokens)
    metrics.record("llm.prompt_tokens", response.usage.prompt_tokens)
    metrics.record("llm.completion_tokens", response.usage.completion_tokens)

    return response.choices[0].message.content

# Make several calls
for i in range(5):
    instrumented_call([{"role": "user", "content": f"Say {i}"}])

# Report
print("Metrics Report:")
for name, summary in metrics.report().items():
    print(f"\n{name}:")
    for k, v in summary.items():
        print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 4. Error Tracking

In [ ]:
class ErrorTracker:
    """Track and analyze errors."""

    def __init__(self):
        self.errors = []

    def capture(self, error: Exception, context: dict = None):
        """Capture an error with context."""
        self.errors.append({
            "type": type(error).__name__,
            "message": str(error),
            "context": context or {},
            "timestamp": datetime.now().isoformat()
        })

    def summary(self) -> dict:
        """Get error summary."""
        from collections import Counter
        error_types = Counter(e["type"] for e in self.errors)

        return {
            "total": len(self.errors),
            "by_type": dict(error_types),
            "recent": self.errors[-5:]
        }

errors = ErrorTracker()

def safe_call(messages: list) -> str:
    """LLM call with error tracking."""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return response.choices[0].message.content
    except Exception as e:
        errors.capture(e, {"messages": messages})
        return f"Error: {e}"

# Test with intentional error
try:
    client.chat.completions.create(model="invalid-model", messages=[])
except Exception as e:
    errors.capture(e, {"model": "invalid-model"})

print(errors.summary())

## 5. Complete Observability Wrapper

In [ ]:
class ObservableLLM:
    """LLM client with full observability."""

    def __init__(self):
        self.client = OpenAI()
        self.tracer = Tracer()
        self.metrics = MetricsCollector()
        self.errors = ErrorTracker()

    def chat(self, messages: list, **kwargs) -> dict:
        """Make a traced, metered LLM call."""

        model = kwargs.get("model", "gpt-4o-mini")

        # Start trace
        trace = self.tracer.start_trace("llm_call", {
            "model": model,
            "message_count": len(messages)
        })

        start = time.time()

        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=messages,
                **{k: v for k, v in kwargs.items() if k != "model"}
            )

            duration = time.time() - start

            # Record metrics
            self.metrics.record("latency", duration, {"model": model})
            self.metrics.record("tokens", response.usage.total_tokens)

            # Update trace
            trace.attributes["tokens"] = response.usage.total_tokens
            trace.finish("success")

            return {
                "content": response.choices[0].message.content,
                "tokens": response.usage.total_tokens,
                "duration": duration
            }

        except Exception as e:
            self.errors.capture(e, {"model": model, "messages": messages})
            trace.finish("error")
            raise

    def get_dashboard(self) -> dict:
        """Get observability dashboard."""
        return {
            "metrics": self.metrics.report(),
            "errors": self.errors.summary(),
            "recent_traces": self.tracer.summary()[-5:]
        }

# Test
observable = ObservableLLM()

for i in range(3):
    result = observable.chat([{"role": "user", "content": f"Count to {i+1}"}])
    print(f"Call {i+1}: {result['tokens']} tokens, {result['duration']:.2f}s")

print("\nDashboard:")
print(json.dumps(observable.get_dashboard(), indent=2, default=str))

## Summary

You learned:
- ✅ Structured logging for AI apps
- ✅ Distributed tracing
- ✅ Metrics collection
- ✅ Error tracking
- ✅ Building observable LLM wrappers

**Next:** [Week 14 - Production Hardening](week_13_production.ipynb)